# 02 — Bronze Layer Test Suite
Tests are organised into three independent suites that share one config registry.
Each suite returns a structured result dict — a final summary cell aggregates all results.

| Suite | What it checks |
|---|---|
| T1 — Volume & Completeness | Row count: source vs Bronze |
| T2 — Row Integrity | SHA-256 fingerprint match: every row, every column |
| T3 — Schema & Metadata | Audit columns, `_rescued_data`, data types |

## Imports

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from typing import Optional

## Source & Table Registry
Single source of truth for all test suites.
Add a new entry here and all three suites pick it up automatically.

In [0]:
CATALOG      = "vstone_catalog"
BRONZE       = "bronze"
CHUNKS_PATH  = f"/Volumes/{CATALOG}/raw/chunks"
LANDING_PATH = f"/Volumes/{CATALOG}/raw/landing"

# Each entry:
#   name            — human-readable label
#   src             — Volume path to raw source file
#   table           — Bronze table name (without catalog/schema prefix)
#   fmt             — Spark reader format
#   opts            — reader options dict
#   null_exclude_col — (optional) exclude source rows where this col is null
#                      used when source file itself contains known bad rows

REGISTRY = [
    {
        "name"             : "Chunk 1 — CSV / COPY INTO",
        "src"              : f"{CHUNKS_PATH}/1_main_chunk_1.csv",
        "table"            : "listings_csv_copyinto",
        "fmt"              : "csv",
        "opts"             : {"header": "true"},
        "null_exclude_col" : "id",   # 2 null-id rows in source — correctly absent from Bronze
    },
    {
        "name"  : "Chunk 2 — CSV / DLT",
        "src"   : f"{CHUNKS_PATH}/1_main_chunk_2.csv",
        "table" : "listings_csv_dlt",
        "fmt"   : "csv",
        "opts"  : {"header": "true"},
    },
    {
        "name"  : "Chunk 3 — JSON / Auto Loader",
        "src"   : f"{CHUNKS_PATH}/1_main_chunk_3.json",
        "table" : "listings_json_autoloader",
        "fmt"   : "json",
        "opts"  : {"multiLine": "true"},
    },
    {
        "name"  : "Chunk 4 — XML / PySpark",
        "src"   : f"{CHUNKS_PATH}/1_main_chunk_4.xml",
        "table" : "listings_xml_pyspark",
        "fmt"   : "xml",
        "opts"  : {"rowTag": "record"},
    },
    {
        "name"  : "Landing — Text Data",
        "src"   : f"{LANDING_PATH}/1_text.csv",
        "table" : "listings_text",
        "fmt"   : "csv",
        "opts"  : {"header": "true", "multiLine": "true", "escape": '"'},
    },
    {
        "name"  : "Landing — Photo Data",
        "src"   : f"{LANDING_PATH}/1_photo.csv",
        "table" : "listings_photo",
        "fmt"   : "csv",
        "opts"  : {"header": "true"},
    },
    {
        "name"  : "Landing — Car Catalog",
        "src"   : f"{LANDING_PATH}/catalogs.csv",
        "table" : "car_catalog",
        "fmt"   : "csv",
        "opts"  : {"header": "true", "sep": ";"},
    },
    {
        "name"  : "Landing — Geo Locations",
        "src"   : f"{LANDING_PATH}/final_geografic.csv",
        "table" : "geo_locations",
        "fmt"   : "csv",
        "opts"  : {"header": "true"},
    },
]

# Collector — all suites append result dicts here
# Each result: {"suite", "name", "status", "detail"}
ALL_RESULTS = []

def full_table(entry):
    """Returns fully-qualified table name from a registry entry."""
    return f"{CATALOG}.{BRONZE}.{entry['table']}"

def read_source(entry):
    """Reads source file with registry options. Applies null_exclude_col if set."""
    df = (spark.read
          .format(entry["fmt"])
          .options(**entry["opts"])
          .option("inferSchema", "false")
          .load(entry["src"]))
    excl_col = entry.get("null_exclude_col")
    if excl_col:
        null_count = df.filter(F.col(excl_col).isNull()).count()
        df = df.filter(F.col(excl_col).isNotNull())
        return df, f"{null_count} null-{excl_col} source rows excluded (bad data in source)"
    return df, None

print(f"Registry loaded — {len(REGISTRY)} tables registered.")

## T1 — Volume & Completeness
Compares row counts between source file and Bronze table.
Source rows with known-bad data (e.g. null id) are excluded before comparison.

In [0]:
def test_volume(entry: dict) -> dict:
    """
    Compares source row count vs Bronze row count.
    Returns a result dict with status PASSED / FAILED.
    """
    try:
        df_src, excl_note = read_source(entry)
        src_count    = df_src.count()
        bronze_count = spark.table(full_table(entry)).count()
        gap          = src_count - bronze_count
        passed       = gap == 0

        detail = f"Raw: {src_count:,} | Bronze: {bronze_count:,}"
        if excl_note:
            detail += f"  NOTE: {excl_note}"
        if not passed:
            detail += f"  GAP: {gap:,}"

        return {"suite": "T1 Volume", "name": entry["name"],
                "status": "PASSED" if passed else "FAILED", "detail": detail}
    except Exception as e:
        return {"suite": "T1 Volume", "name": entry["name"],
                "status": "ERROR", "detail": str(e)}


# ── Run T1 ──
print("T1 — VOLUME & COMPLETENESS")
print("-" * 75)
for entry in REGISTRY:
    result = test_volume(entry)
    ALL_RESULTS.append(result)
    print(f"  {result['status']:<6} | {result['name']:<35} | {result['detail']}")
print("-" * 75)

## T2 — Row-to-Row Integrity
Generates a SHA-256 fingerprint for every row using all columns present in both
source and Bronze. Compares fingerprint sets via subtract — any mismatch means
a row was corrupted, altered, or missing.

In [0]:
def _fingerprint_df(df, columns):
    """
    Normalises each column to trimmed STRING, replaces NULLs with '',
    then hashes all columns together into a single SHA-256 fingerprint per row.
    """
    normalised = df.select([
        F.coalesce(F.trim(F.col(c).cast("string")), F.lit("")).alias(c)
        for c in columns
    ])
    return (normalised
            .withColumn("fingerprint", F.sha2(F.concat_ws("||", *columns), 256))
            .select("fingerprint"))


def test_integrity(entry: dict) -> dict:
    """
    Compares SHA-256 row fingerprints between source and Bronze.
    Only columns present in both DataFrames are included in the hash.
    Returns a result dict with status PASSED / FAILED.
    """
    try:
        df_src, excl_note  = read_source(entry)
        df_brz             = spark.table(full_table(entry))
        common_cols        = [c for c in df_src.columns if c in df_brz.columns]

        src_fp = _fingerprint_df(df_src, common_cols)
        brz_fp = _fingerprint_df(df_brz, common_cols)

        missing = src_fp.subtract(brz_fp).count()   # in source, not in Bronze
        extra   = brz_fp.subtract(src_fp).count()   # in Bronze, not in source
        total   = missing + extra
        passed  = total == 0

        detail = f"Columns compared: {len(common_cols)}"
        if excl_note:
            detail += f"  NOTE: {excl_note}"
        if not passed:
            detail += f"  Missing in Bronze: {missing} | Extra in Bronze: {extra}"

        return {"suite": "T2 Integrity", "name": entry["name"],
                "status": "PASSED" if passed else "FAILED", "detail": detail}
    except Exception as e:
        return {"suite": "T2 Integrity", "name": entry["name"],
                "status": "ERROR", "detail": str(e)}


# ── Run T2 ──
print("T2 — ROW-TO-ROW INTEGRITY (SHA-256 FINGERPRINT)")
print("-" * 75)
for entry in REGISTRY:
    result = test_integrity(entry)
    ALL_RESULTS.append(result)
    print(f"  {result['status']:<6} | {result['name']:<35} | {result['detail']}")
print("-" * 75)

## T3 — Schema & Metadata
Three checks per Bronze table:
1. **Audit columns** — `load_dt` and `source_file` must exist and be 100% populated
2. **`_rescued_data`** — if present, must be all-null (no data loss from schema mismatch)
3. **Data types** — `load_dt` must be TIMESTAMP, all source columns must be STRING

In [0]:
def test_schema_and_metadata(entry: dict) -> dict:
    """
    Validates audit columns, _rescued_data cleanliness, and column data types
    for a Bronze table. Returns a result dict with status PASSED / FAILED.
    """
    issues = []
    notes  = []
    try:
        df   = spark.table(full_table(entry))
        cols = df.columns
        dtypes = dict(df.dtypes)

        # ── Check 1: Audit columns exist and are fully populated ──
        for audit_col in ("load_dt", "source_file"):
            if audit_col not in cols:
                issues.append(f"{audit_col} column missing")
            else:
                null_count = df.filter(F.col(audit_col).isNull()).count()
                if null_count > 0:
                    issues.append(f"{audit_col} has {null_count:,} null rows")

        # ── Check 2: load_dt must be TIMESTAMP ──
        if "load_dt" in dtypes and not dtypes["load_dt"].startswith("timestamp"):
            issues.append(f"load_dt is {dtypes['load_dt']}, expected timestamp")

        # ── Check 3: _rescued_data must be all-null if present ──
        if "_rescued_data" in cols:
            rescued = df.filter(F.col("_rescued_data").isNotNull()).count()
            if rescued > 0:
                issues.append(f"_rescued_data has {rescued:,} non-null rows (schema mismatch)")
            else:
                notes.append("_rescued_data present but clean")

        passed = len(issues) == 0
        detail = "All checks passed"
        if notes:
            detail += "  |  " + " | ".join(notes)
        if issues:
            detail = " | ".join(issues)

        return {"suite": "T3 Schema", "name": entry["name"],
                "status": "PASSED" if passed else "FAILED", "detail": detail}
    except Exception as e:
        return {"suite": "T3 Schema", "name": entry["name"],
                "status": "ERROR", "detail": str(e)}


# ── Run T3 ──
print("T3 — SCHEMA & METADATA AUDIT")
print("-" * 75)
for entry in REGISTRY:
    result = test_schema_and_metadata(entry)
    ALL_RESULTS.append(result)
    print(f"  {result['status']:<6} | {result['name']:<35} | {result['detail']}")
print("-" * 75)

## Final Summary
Aggregates results from all three suites into one report.
Any FAILED or ERROR rows are printed first for immediate visibility.

In [0]:
def print_summary(results: list):
    """
    Prints a consolidated pass/fail report grouped by suite.
    Exits with a clear OVERALL PASSED / FAILED line.
    """
    suites   = ["T1 Volume", "T2 Integrity", "T3 Schema"]
    total    = len(results)
    failures = [r for r in results if r["status"] != "PASSED"]

    print("=" * 75)
    print("  BRONZE LAYER TEST SUITE — FINAL SUMMARY")
    print("=" * 75)

    for suite in suites:
        suite_results = [r for r in results if r["suite"] == suite]
        passed  = sum(1 for r in suite_results if r["status"] == "PASSED")
        total_s = len(suite_results)
        print(f"  {suite:<18} : {passed}/{total_s} passed")

    print("-" * 75)

    if failures:
        print(f"  FAILURES / ERRORS ({len(failures)}):")
        for r in failures:
            print(f"    [{r['suite']}] {r['name']}")
            print(f"      {r['status']}: {r['detail']}")
    else:
        print("  No failures.")

    print("=" * 75)
    overall = "ALL TESTS PASSED" if not failures else f"{len(failures)} TEST(S) FAILED"
    print(f"  OVERALL: {overall}  ({total - len(failures)}/{total} checks passed)")
    print("=" * 75)


print_summary(ALL_RESULTS)